# ⚽ FIFA Dataset Analysis - Web Scraping Project

This notebook demonstrates:
- Web scraping FIFA player data using `requests` and `BeautifulSoup`
- Data cleaning and preprocessing with `pandas`
- Exploratory Data Analysis (EDA)
- Visualizations using `matplotlib` and `seaborn`

## 1. Install & Import Libraries

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ All libraries imported successfully!')

## 2. Web Scraping FIFA Player Data

We scrape player data from **sofifa.com** — a popular public FIFA stats site.

In [ ]:
def get_headers():
    """Return browser-like headers to avoid being blocked."""
    return {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        ),
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept-Encoding': 'gzip, deflate, br',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Connection': 'keep-alive',
    }

BASE_URL = 'https://sofifa.com/players'

def scrape_page(page_offset: int = 0) -> list[dict]:
    """
    Scrape one page of player listings from sofifa.com.
    Returns a list of player dicts.
    """
    url = f'{BASE_URL}?offset={page_offset}'
    try:
        resp = requests.get(url, headers=get_headers(), timeout=15)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f'  ⚠️  Request failed for offset {page_offset}: {e}')
        return []

    soup = BeautifulSoup(resp.text, 'lxml')
    rows = soup.select('table tbody tr')
    players = []

    for row in rows:
        cols = row.find_all('td')
        if len(cols) < 8:
            continue
        try:
            name_tag = cols[1].find('a')
            name = name_tag.get_text(strip=True) if name_tag else 'Unknown'

            # Position badges
            pos_tags = cols[2].find_all('a')
            positions = ', '.join(p.get_text(strip=True) for p in pos_tags) if pos_tags else 'N/A'

            age    = cols[3].get_text(strip=True)
            overall = cols[4].get_text(strip=True)
            potential = cols[5].get_text(strip=True)

            # Club & nationality from img alt tags
            imgs = cols[1].find_all('img')
            nationality = imgs[0]['title'] if len(imgs) > 0 and imgs[0].has_attr('title') else 'N/A'
            club        = imgs[1]['title'] if len(imgs) > 1 and imgs[1].has_attr('title') else 'N/A'

            # Value & wage
            value = cols[6].get_text(strip=True)
            wage  = cols[7].get_text(strip=True)

            players.append({
                'Name': name,
                'Nationality': nationality,
                'Club': club,
                'Positions': positions,
                'Age': age,
                'Overall': overall,
                'Potential': potential,
                'Value': value,
                'Wage': wage,
            })
        except Exception:
            continue

    return players

print('✅ Scraping functions defined.')

In [ ]:
# ── Scrape multiple pages ──────────────────────────────────────────────────────
# sofifa paginates in steps of 60. Scrape 5 pages = ~300 players.
# Increase `num_pages` for a larger dataset (be polite — add delays).

NUM_PAGES   = 5
PAGE_STEP   = 60
all_players = []

for page in range(NUM_PAGES):
    offset = page * PAGE_STEP
    print(f'Scraping page {page + 1}/{NUM_PAGES} (offset={offset}) …', end=' ')
    players = scrape_page(offset)
    all_players.extend(players)
    print(f'{len(players)} players fetched.')
    time.sleep(1.5)   # be a polite scraper

print(f'\n✅ Total players scraped: {len(all_players)}')

## 3. Build the DataFrame & Clean the Data

In [ ]:
df = pd.DataFrame(all_players)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
def parse_currency(val: str) -> float:
    """
    Convert sofifa currency strings like '€120.5M', '€750K', '€0' → float (EUR).
    """
    val = str(val).replace('€', '').replace(',', '').strip()
    if val.endswith('M'):
        return float(val[:-1]) * 1_000_000
    elif val.endswith('K'):
        return float(val[:-1]) * 1_000
    else:
        try:
            return float(val)
        except ValueError:
            return np.nan

# Numeric conversions
df['Age']       = pd.to_numeric(df['Age'],       errors='coerce')
df['Overall']   = pd.to_numeric(df['Overall'],   errors='coerce')
df['Potential'] = pd.to_numeric(df['Potential'], errors='coerce')
df['Value_EUR'] = df['Value'].apply(parse_currency)
df['Wage_EUR']  = df['Wage'].apply(parse_currency)

# Derived column
df['Growth'] = df['Potential'] - df['Overall']

# Drop duplicates & reset index
df.drop_duplicates(subset='Name', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Clean shape: {df.shape}')
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
df[['Name','Nationality','Club','Positions','Age','Overall','Potential',
    'Growth','Value_EUR','Wage_EUR']].head(10)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Basic Statistics ===')
df[['Age', 'Overall', 'Potential', 'Growth', 'Value_EUR', 'Wage_EUR']].describe().round(2)

In [ ]:
# ── Top 10 players by Overall rating ──────────────────────────────────────────
top10 = df.nlargest(10, 'Overall')[['Name','Nationality','Club','Overall','Potential','Value_EUR']]
print('🏆 Top 10 Players by Overall Rating')
top10

In [ ]:
# ── Top nationalities ─────────────────────────────────────────────────────────
top_nations = df['Nationality'].value_counts().head(15)
print('🌍 Top 15 Nationalities')
print(top_nations.to_string())

## 5. Visualizations

In [ ]:
# ── 5.1  Overall Rating Distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Overall'].dropna(), bins=20, kde=True,
             color='steelblue', edgecolor='white', ax=axes[0])
axes[0].set_title('Distribution of Overall Ratings', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Overall Rating')
axes[0].set_ylabel('Count')

sns.boxplot(y=df['Overall'].dropna(), color='lightcoral', ax=axes[1])
axes[1].set_title('Overall Rating — Box Plot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Overall Rating')

plt.tight_layout()
plt.savefig('overall_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Figure saved as overall_distribution.png')

In [ ]:
# ── 5.2  Age vs Overall Rating ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(
    df['Age'], df['Overall'],
    c=df['Value_EUR'], cmap='viridis',
    alpha=0.7, s=50, edgecolors='none'
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Market Value (€)', fontsize=11)
ax.set_title('Age vs Overall Rating (coloured by Market Value)', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Overall Rating')
plt.tight_layout()
plt.savefig('age_vs_overall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.3  Top 10 Players by Market Value ──────────────────────────────────────
top_value = df.nlargest(10, 'Value_EUR')[['Name', 'Value_EUR']].sort_values('Value_EUR')

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_value['Name'], top_value['Value_EUR'] / 1e6,
               color=sns.color_palette('Blues_d', len(top_value)))
ax.set_xlabel('Market Value (€ Millions)')
ax.set_title('Top 10 Players by Market Value', fontsize=14, fontweight='bold')
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'€{bar.get_width():.1f}M', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('top_value_players.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4  Top Nationalities ────────────────────────────────────────────────────
top_nations = df['Nationality'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top_nations.values, y=top_nations.index,
            palette='rocket_r', ax=ax)
ax.set_title('Top 15 Nationalities in Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Players')
ax.set_ylabel('Nationality')
plt.tight_layout()
plt.savefig('top_nationalities.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.5  Overall vs Potential (Growth potential heat) ────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
sns.kdeplot(
    data=df.dropna(subset=['Overall', 'Potential']),
    x='Overall', y='Potential',
    fill=True, cmap='YlOrRd', thresh=0.02, ax=ax
)
# Diagonal line: potential == overall (no growth)
lims = [max(df['Overall'].min(), df['Potential'].min()),
        min(df['Overall'].max(), df['Potential'].max())]
ax.plot(lims, lims, 'k--', linewidth=1, label='No growth line')
ax.set_title('Overall vs Potential Rating Density', fontsize=14, fontweight='bold')
ax.set_xlabel('Overall Rating')
ax.set_ylabel('Potential Rating')
ax.legend()
plt.tight_layout()
plt.savefig('overall_vs_potential.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.6  Age Distribution by Overall Band ────────────────────────────────────
bins   = [0, 65, 74, 79, 84, 100]
labels = ['<65','65–74','75–79','80–84','85+']
df['Rating_Band'] = pd.cut(df['Overall'], bins=bins, labels=labels, right=True)

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df.dropna(subset=['Age','Rating_Band']),
               x='Rating_Band', y='Age',
               palette='Set2', inner='quartile', ax=ax)
ax.set_title('Age Distribution by Overall Rating Band', fontsize=14, fontweight='bold')
ax.set_xlabel('Overall Rating Band')
ax.set_ylabel('Age')
plt.tight_layout()
plt.savefig('age_by_rating_band.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

In [ ]:
numeric_cols = ['Age', 'Overall', 'Potential', 'Growth', 'Value_EUR', 'Wage_EUR']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5, ax=ax
)
ax.set_title('Correlation Matrix of Key Numeric Variables',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Cleaned Data

In [ ]:
output_cols = ['Name','Nationality','Club','Positions','Age',
               'Overall','Potential','Growth','Value_EUR','Wage_EUR']
df[output_cols].to_csv('fifa_players_cleaned.csv', index=False)
print(f'✅ Saved fifa_players_cleaned.csv  ({len(df)} rows)')

## 8. Key Findings

| Metric | Value |
|--------|-------|
| Total players scraped | `df.shape[0]` |
| Average overall rating | `df['Overall'].mean().round(1)` |
| Average age | `df['Age'].mean().round(1)` |
| Most represented nation | `df['Nationality'].value_counts().idxmax()` |
| Highest-value player | `df.loc[df['Value_EUR'].idxmax(), 'Name']` |

> **Run the cell below to print a live summary.**

In [ ]:
print('═' * 50)
print('          ⚽  FIFA SCRAPING SUMMARY  ⚽')
print('═' * 50)
print(f'Total players scraped  : {len(df)}')
print(f'Avg Overall Rating     : {df["Overall"].mean():.1f}')
print(f'Avg Age                : {df["Age"].mean():.1f} years')
top_nation = df['Nationality'].value_counts().idxmax()
top_count  = df['Nationality'].value_counts().max()
print(f'Most represented nation: {top_nation} ({top_count} players)')
best_player = df.loc[df['Value_EUR'].idxmax()]
print(f'Highest market value   : {best_player["Name"]} '
      f'(€{best_player["Value_EUR"]/1e6:.1f}M)')
print('═' * 50)